# `WC_BADGE_DETAILS_D` Incremental Load

- Source File: `SCEN_TASK_NO` based ODI script
- Conversion Date: 2023-10-27

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType, DoubleType,
    DecimalType, TimestampType, DateType, BinaryType, FloatType
)
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
dbutils.widgets.text("DATASOURCE_NUM_ID", "380") # Default to 380 as seen in source SQL
dbutils.widgets.text("ETL_PROC_WID",      "")
dbutils.widgets.text("ODI_SESS_NO",       "")
dbutils.widgets.text("v_ETL_JOB_TYPE",    "WC_BADGE_DETAILS_D_INCREMENTAL") # From #GLOBAL.v_ETL_JOB_TYPE

datasource_num_id = int(dbutils.widgets.get("DATASOURCE_NUM_ID"))
etl_proc_wid      = int(dbutils.widgets.get("ETL_PROC_WID")) if dbutils.widgets.get("ETL_PROC_WID") else None # Handle potential empty string
odi_sess_no       = dbutils.widgets.get("ODI_SESS_NO")
v_etl_job_type    = dbutils.widgets.get("v_ETL_JOB_TYPE")

## ETL Parameters

In [ ]:
# SCEN_TASK_NO {2}, {3}, {4}, {5}, {6}
# Fetch ETL extract times and process WID from wc_etl_parameters

etl_params_df = (
    spark.table("workspace.prxbi_dw.wc_etl_parameters")
    .filter(F.col("etl_job_type") == v_etl_job_type)
    .select("etl_last_extract_time", "etl_current_extract_time", "row_wid")
    .collect()[0]
)

etl_last_extract_time   = etl_params_df["etl_last_extract_time"]
etl_current_extract_time = etl_params_df["etl_current_extract_time"]
# If etl_proc_wid was not provided as a widget, use the one from wc_etl_parameters
if etl_proc_wid is None:
    etl_proc_wid = etl_params_df["row_wid"]

print(f"ETL Last Extract Time: {etl_last_extract_time}")
print(f"ETL Current Extract Time: {etl_current_extract_time}")
print(f"ETL Process WID: {etl_proc_wid}")

## Staging Table: `c_mercury_badge_stg`

In [ ]:
# SCEN_TASK_NO {30}
# Drop the staging table if it exists
spark.sql("DROP TABLE IF EXISTS workspace.prxbi_dw.c_mercury_badge_stg PURGE")

In [ ]:
# SCEN_TASK_NO {40}, {50}
# Build and write data to the staging table
# This includes the ODI MAX self-join dedup pattern, converted to a Window function.

# Deduplicate WC_MERCURY_BADGE_TS based on INT_INSERT_DATE and VERSIONNUMBER for each ID
window_spec_source = Window.partitionBy("ID").orderBy(F.col("INT_INSERT_DATE").desc(), F.col("VERSIONNUMBER").desc())

source_deduped_df = (
    spark.table("workspace.prxbi_ts.wc_mercury_badge_ts")
    .filter(
        (F.col("INT_INSERT_DATE") > F.lit(etl_last_extract_time))
        & (F.col("INT_INSERT_DATE") <= F.lit(etl_current_extract_time))
    )
    .withColumn("rn", F.row_number().over(window_spec_source))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Select and cast columns for the staging table
staging_df = source_deduped_df.select(
    F.col("ID").cast("string"),
    F.col("BADGELOCATION").cast("string"),
    F.col("BADGETOKEN").cast("string"),
    F.col("BADGEVERSION").cast("long"),
    F.col("CONTACTEMAIL").cast("string"),
    F.col("CONTACTFIRSTNAME").cast("string"),
    F.col("CONTACTJOBTITLE").cast("string"),
    F.col("CONTACTLASTNAME").cast("string"),
    F.col("CONTACTPERSONRXMASTERID").cast("string"),
    F.col("CREATEDBYREGISTRATIONTYPE").cast("string"),
    F.col("CREATEDBYTYPE").cast("string"),
    F.col("CULTURE").cast("string"),
    F.col("CUSTOMERTYPE").cast("string"),
    F.col("EVENTEDITIONGBSCODE").cast("string"),
    F.col("ISBADGEUPDATE").cast("string"),
    F.col("MARKETINGPREFERENCESPROMPTREQUIRED").alias("MARKETINGPREFERENCESPROMPTREQU").cast("string"), # Renaming as per target
    F.col("ORGANISATIONCITY").cast("string"),
    F.col("ORGANISATIONCOUNTRYCODE").cast("string"),
    F.col("ORGANISATIONDISPLAYNAME").cast("string"),
    F.col("ORGANISATIONRXMASTERID").cast("string"),
    F.col("ORGANISATIONSTATE").cast("string"),
    F.col("PARTICIPATINGORGANISATIONID").cast("string"),
    F.col("PRODUCTCODE").cast("string"),
    F.col("QRCODECONTENT").cast("string"),
    F.col("REGISTRATIONID").cast("string"),
    F.col("STATUS").cast("long"),
    F.col("SUPPORTSTAFFCOMPANYADDRESS").cast("string"),
    F.col("SUPPORTSTAFFCOMPANYNAME").cast("string"),
    F.col("SUPPORTSTAFFMOBILEPHONE").cast("string"),
    F.col("SUPPORTSTAFFREPORTSTO").cast("string"),
    F.col("SUPPORTSTAFFSTANDS").cast("string"),
    F.col("SUPPORTSTAFFUSERACCESS").cast("string"),
    F.col("VERSIONNUMBER").cast("long"),
    F.col("MOBILEPHONE").cast("string"),
    F.col("FIRSTSCANNEDDATE").cast("timestamp"),
    F.col("LASTPRINTEDDATE").cast("timestamp"),
    F.col("ACCESSVALIDITYMODIFIEDDATE").cast("timestamp"),
    F.col("CREATEDDATE").cast("timestamp"),
    F.col("COMPANYPRODUCTCODE").cast("string"),
    F.col("PAYMENTSTATUS").cast("string"),
    F.col("PHOTOKEY").cast("string"),
    F.col("PHOTOSOURCE").cast("string"),
    F.col("PHOTOSOURCETYPE").cast("string")
)

( 
    staging_df
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") # Added due to potential schema evolution with implicit creation
    .saveAsTable("workspace.prxbi_dw.c_mercury_badge_stg")
)

In [ ]:
c_mercury_badge_stg_count = spark.table("workspace.prxbi_dw.c_mercury_badge_stg").count()
print(f"Staging count: {c_mercury_badge_stg_count}")

## Optimize Staging Table

In [ ]:
# SCEN_TASK_NO {60}
# Optimize staging table (replaces DBMS_STATS)
spark.sql("SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false")
spark.sql("OPTIMIZE workspace.prxbi_dw.c_mercury_badge_stg ZORDER BY (id)")

## Flow Table: `i_wc_badge_details_d_flow`

In [ ]:
# SCEN_TASK_NO {80}
# Drop the flow table if it exists
spark.sql("DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow")

In [ ]:
# SCEN_TASK_NO {90}, {100}
# Build and write data to the flow table

# Deduplicate WC_BADGE_PRODUCT_D using a Window function (replaces rank() over)
window_spec_product = Window.partitionBy("SKU").orderBy(F.col("ID").desc())

wc_badge_product_d_deduped = (
    spark.table("workspace.prxbi_dw.wc_badge_product_d")
    .withColumn("rn", F.rank().over(window_spec_product))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        F.col("SKU").alias("SKU_1"),
        F.col("NAME").alias("NAME_1")
    )
)

# Main transformation logic for the flow table
flow_df_source = (
    spark.table("workspace.prxbi_dw.c_mercury_badge_stg").alias("join1_a")
    .join(
        wc_badge_product_d_deduped.alias("wc_badge_product_d_2"),
        F.col("join1_a.PRODUCTCODE") == F.col("wc_badge_product_d_2.SKU_1"),
        "left_outer"
    )
    .select(
        F.col("join1_a.ID").alias("BADGE_ID"),
        F.col("join1_a.BADGELOCATION").alias("BADGE_LOCATION"),
        F.col("join1_a.BADGETOKEN").alias("BADGE_TOKEN"),
        F.col("join1_a.BADGEVERSION").alias("BADGE_VERSION"),
        F.col("join1_a.CONTACTEMAIL").alias("CONTACT_EMAIL"),
        F.col("join1_a.CONTACTFIRSTNAME").alias("CONTACT_FIRST_NAME"),
        F.col("join1_a.CONTACTLASTNAME").alias("CONTACT_LAST_NAME"),
        F.col("join1_a.CONTACTJOBTITLE").alias("CONTACT_JOB_TITLE"),
        F.col("join1_a.CONTACTPERSONRXMASTERID").alias("CONTACT_PERSON_ID"),
        F.col("join1_a.CREATEDBYREGISTRATIONTYPE").alias("CREATION_REG_TYPE"),
        F.col("join1_a.CREATEDBYTYPE").alias("CREATION_TYPE"),
        F.col("join1_a.CULTURE").alias("CULTURE"),
        F.col("join1_a.CUSTOMERTYPE").alias("CUSTOMER_TYPE"),
        F.col("join1_a.EVENTEDITIONGBSCODE").alias("EVENT_EDITION_CODE"),
        F.col("join1_a.ISBADGEUPDATE").alias("BADGE_UPDATE_FLG"),
        F.col("join1_a.MARKETINGPREFERENCESPROMPTREQU").alias("MARKETING_PREF_PROMPT"),
        F.col("join1_a.ORGANISATIONDISPLAYNAME").alias("ORG_NAME"),
        F.col("join1_a.ORGANISATIONCITY").alias("ORG_CITY"),
        F.col("join1_a.ORGANISATIONCOUNTRYCODE").alias("ORG_COUNTRY"),
        F.col("join1_a.ORGANISATIONRXMASTERID").alias("ORG_ID"),
        F.col("join1_a.ORGANISATIONSTATE").alias("ORG_STATE"),
        F.col("join1_a.PARTICIPATINGORGANISATIONID").alias("PARTICIPATING_ORG_ID"),
        F.col("join1_a.PRODUCTCODE").alias("PRODUCT_CODE"),
        F.col("join1_a.QRCODECONTENT").alias("QR_CODE"),
        F.col("join1_a.REGISTRATIONID").alias("REGISTRATION_ID"),
        F.col("join1_a.STATUS").alias("STATUS"),
        F.col("join1_a.SUPPORTSTAFFCOMPANYNAME").alias("STAFF_COMPANY_NAME"),
        F.col("join1_a.SUPPORTSTAFFCOMPANYADDRESS").alias("STAFF_COMPANY_ADDR"),
        F.col("join1_a.SUPPORTSTAFFMOBILEPHONE").alias("STAFF_PHONE_NUM"),
        F.col("join1_a.SUPPORTSTAFFREPORTSTO").alias("STAFF_REPORTING"),
        F.col("join1_a.SUPPORTSTAFFSTANDS").alias("STAFF_STANDS"),
        F.col("join1_a.SUPPORTSTAFFUSERACCESS").alias("STAFF_USER_ACCESS"),
        F.col("join1_a.VERSIONNUMBER").alias("VERSION_NUM"),
        F.col("join1_a.ID").alias("INTEGRATION_ID"),
        F.lit(datasource_num_id).alias("DATASOURCE_NUM_ID"), # Using widget parameter
        F.col("join1_a.MOBILEPHONE").alias("MOBILEPHONE"),
        F.col("join1_a.FIRSTSCANNEDDATE").alias("FIRSTSCANNEDDATE"),
        F.col("join1_a.LASTPRINTEDDATE").alias("LASTPRINTEDDATE"),
        F.when(F.col("join1_a.FIRSTSCANNEDDATE").isNotNull(), F.lit("Y")).otherwise(F.lit("N")).alias("FIRSTSCANNEDDATE_FLG"),
        F.when(F.col("join1_a.LASTPRINTEDDATE").isNotNull(), F.lit("Y")).otherwise(F.lit("N")).alias("LASTPRINTEDDATE_FLG"),
        F.col("join1_a.ACCESSVALIDITYMODIFIEDDATE").alias("ACCESSVALIDITYMODIFIEDDATE"),
        F.col("join1_a.CREATEDDATE").alias("CREATEDDATE"),
        F.col("join1_a.COMPANYPRODUCTCODE").alias("COMPANYPRODUCTCODE"),
        F.col("join1_a.PAYMENTSTATUS").alias("PAYMENTSTATUS"),
        F.col("join1_a.PHOTOKEY").alias("PHOTOKEY"),
        F.col("join1_a.PHOTOSOURCE").alias("PHOTOSOURCE"),
        F.col("join1_a.PHOTOSOURCETYPE").alias("PHOTOSOURCETYPE"),
        F.col("wc_badge_product_d_2.NAME_1").alias("PACKAGE_NAME"),
        F.lit("I").alias("IND_UPDATE")
    )
)

# The NOT EXISTS clause in the original SQL is handled by the main merge operation later.
# For now, we just create the full flow table content based on the source joins.
( 
    flow_df_source
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.prxbi_dw.i_wc_badge_details_d_flow")
)

In [ ]:
i_wc_badge_details_d_flow_count = spark.table("workspace.prxbi_dw.i_wc_badge_details_d_flow").count()
print(f"Flow count: {i_wc_badge_details_d_flow_count}")

## Optimize Flow Table

In [ ]:
# SCEN_TASK_NO {110}, {120}
# Optimize flow table (replaces create index and DBMS_STATS)
spark.sql("SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false")
spark.sql("OPTIMIZE workspace.prxbi_dw.i_wc_badge_details_d_flow ZORDER BY (integration_id, datasource_num_id)")

## Mark Records for Update in Flow Table

In [ ]:
# SCEN_TASK_NO {130}
# Update IND_UPDATE flag for records already existing in the target table
DeltaTable.forName(spark, "workspace.prxbi_dw.i_wc_badge_details_d_flow").alias("t").merge(
    spark.table("workspace.prxbi_dw.wc_badge_details_d")
    .select("INTEGRATION_ID", "DATASOURCE_NUM_ID").distinct().alias("s"),
    "t.INTEGRATION_ID = s.INTEGRATION_ID AND t.DATASOURCE_NUM_ID = s.DATASOURCE_NUM_ID"
).whenMatchedUpdate(set={
    "t.IND_UPDATE": F.lit("U")
}).execute()

## Merge into Target: `wc_badge_details_d`

In [ ]:
# SCEN_TASK_NO {150}, {160}
# Merge data from the flow table into the target table

# Note: ROW_WID is excluded from merge as it's typically an IDENTITY column.
# The original SQL uses WC_BADGE_DETAILS_D_SEQ.NEXTVAL for insert, confirming it's generated.
# W_INSERT_DT and W_UPDATE_DT are managed by the merge operation.

target_table = DeltaTable.forName(spark, "workspace.prxbi_dw.wc_badge_details_d")
flow_table_df = spark.table("workspace.prxbi_dw.i_wc_badge_details_d_flow")

target_table.alias("t").merge(
    flow_table_df.alias("s"),
    "t.INTEGRATION_ID = s.INTEGRATION_ID AND t.DATASOURCE_NUM_ID = s.DATASOURCE_NUM_ID"
)
.whenMatchedUpdate(condition="s.IND_UPDATE = 'U'", set={
    "t.BADGE_ID":                     "s.BADGE_ID",
    "t.BADGE_LOCATION":               "s.BADGE_LOCATION",
    "t.BADGE_TOKEN":                  "s.BADGE_TOKEN",
    "t.BADGE_VERSION":                "s.BADGE_VERSION",
    "t.CONTACT_EMAIL":                "s.CONTACT_EMAIL",
    "t.CONTACT_FIRST_NAME":           "s.CONTACT_FIRST_NAME",
    "t.CONTACT_LAST_NAME":            "s.CONTACT_LAST_NAME",
    "t.CONTACT_JOB_TITLE":            "s.CONTACT_JOB_TITLE",
    "t.CONTACT_PERSON_ID":            "s.CONTACT_PERSON_ID",
    "t.CREATION_REG_TYPE":            "s.CREATION_REG_TYPE",
    "t.CREATION_TYPE":                "s.CREATION_TYPE",
    "t.CULTURE":                      "s.CULTURE",
    "t.CUSTOMER_TYPE":                "s.CUSTOMER_TYPE",
    "t.EVENT_EDITION_CODE":           "s.EVENT_EDITION_CODE",
    "t.BADGE_UPDATE_FLG":             "s.BADGE_UPDATE_FLG",
    "t.MARKETING_PREF_PROMPT":        "s.MARKETING_PREF_PROMPT",
    "t.ORG_NAME":                     "s.ORG_NAME",
    "t.ORG_CITY":                     "s.ORG_CITY",
    "t.ORG_COUNTRY":                  "s.ORG_COUNTRY",
    "t.ORG_ID":                       "s.ORG_ID",
    "t.ORG_STATE":                    "s.ORG_STATE",
    "t.PARTICIPATING_ORG_ID":         "s.PARTICIPATING_ORG_ID",
    "t.PRODUCT_CODE":                 "s.PRODUCT_CODE",
    "t.QR_CODE":                      "s.QR_CODE",
    "t.REGISTRATION_ID":              "s.REGISTRATION_ID",
    "t.STATUS":                       "s.STATUS",
    "t.STAFF_COMPANY_NAME":           "s.STAFF_COMPANY_NAME",
    "t.STAFF_COMPANY_ADDR":           "s.STAFF_COMPANY_ADDR",
    "t.STAFF_PHONE_NUM":              "s.STAFF_PHONE_NUM",
    "t.STAFF_REPORTING":              "s.STAFF_REPORTING",
    "t.STAFF_STANDS":                 "s.STAFF_STANDS",
    "t.STAFF_USER_ACCESS":            "s.STAFF_USER_ACCESS",
    "t.VERSION_NUM":                  "s.VERSION_NUM",
    "t.MOBILEPHONE":                  "s.MOBILEPHONE",
    "t.FIRSTSCANNEDDATE":             "s.FIRSTSCANNEDDATE",
    "t.LASTPRINTEDDATE":              "s.LASTPRINTEDDATE",
    "t.FIRSTSCANNEDDATE_FLG":         "s.FIRSTSCANNEDDATE_FLG",
    "t.LASTPRINTEDDATE_FLG":          "s.LASTPRINTEDDATE_FLG",
    "t.ACCESSVALIDITYMODIFIEDDATE":   "s.ACCESSVALIDITYMODIFIEDDATE",
    "t.CREATEDDATE":                  "s.CREATEDDATE",
    "t.COMPANYPRODUCTCODE":           "s.COMPANYPRODUCTCODE",
    "t.PAYMENTSTATUS":                "s.PAYMENTSTATUS",
    "t.PHOTOKEY":                     "s.PHOTOKEY",
    "t.PHOTOSOURCE":                  "s.PHOTOSOURCE",
    "t.PHOTOSOURCETYPE":              "s.PHOTOSOURCETYPE",
    "t.PACKAGE_NAME":                 "s.PACKAGE_NAME",
    "t.W_UPDATE_DT":                  F.current_timestamp()
})
.whenNotMatchedInsert(condition="s.IND_UPDATE = 'I'", values={
    "BADGE_ID":                     "s.BADGE_ID",
    "BADGE_LOCATION":               "s.BADGE_LOCATION",
    "BADGE_TOKEN":                  "s.BADGE_TOKEN",
    "BADGE_VERSION":                "s.BADGE_VERSION",
    "CONTACT_EMAIL":                "s.CONTACT_EMAIL",
    "CONTACT_FIRST_NAME":           "s.CONTACT_FIRST_NAME",
    "CONTACT_LAST_NAME":            "s.CONTACT_LAST_NAME",
    "CONTACT_JOB_TITLE":            "s.CONTACT_JOB_TITLE",
    "CONTACT_PERSON_ID":            "s.CONTACT_PERSON_ID",
    "CREATION_REG_TYPE":            "s.CREATION_REG_TYPE",
    "CREATION_TYPE":                "s.CREATION_TYPE",
    "CULTURE":                      "s.CULTURE",
    "CUSTOMER_TYPE":                "s.CUSTOMER_TYPE",
    "EVENT_EDITION_CODE":           "s.EVENT_EDITION_CODE",
    "BADGE_UPDATE_FLG":             "s.BADGE_UPDATE_FLG",
    "MARKETING_PREF_PROMPT":        "s.MARKETING_PREF_PROMPT",
    "ORG_NAME":                     "s.ORG_NAME",
    "ORG_CITY":                     "s.ORG_CITY",
    "ORG_COUNTRY":                  "s.ORG_COUNTRY",
    "ORG_ID":                       "s.ORG_ID",
    "ORG_STATE":                    "s.ORG_STATE",
    "PARTICIPATING_ORG_ID":         "s.PARTICIPATING_ORG_ID",
    "PRODUCT_CODE":                 "s.PRODUCT_CODE",
    "QR_CODE":                      "s.QR_CODE",
    "REGISTRATION_ID":              "s.REGISTRATION_ID",
    "STATUS":                       "s.STATUS",
    "STAFF_COMPANY_NAME":           "s.STAFF_COMPANY_NAME",
    "STAFF_COMPANY_ADDR":           "s.STAFF_COMPANY_ADDR",
    "STAFF_PHONE_NUM":              "s.STAFF_PHONE_NUM",
    "STAFF_REPORTING":              "s.STAFF_REPORTING",
    "STAFF_STANDS":                 "s.STAFF_STANDS",
    "STAFF_USER_ACCESS":            "s.STAFF_USER_ACCESS",
    "VERSION_NUM":                  "s.VERSION_NUM",
    "INTEGRATION_ID":               "s.INTEGRATION_ID",
    "DATASOURCE_NUM_ID":            "s.DATASOURCE_NUM_ID",
    "MOBILEPHONE":                  "s.MOBILEPHONE",
    "FIRSTSCANNEDDATE":             "s.FIRSTSCANNEDDATE",
    "LASTPRINTEDDATE":              "s.LASTPRINTEDDATE",
    "FIRSTSCANNEDDATE_FLG":         "s.FIRSTSCANNEDDATE_FLG",
    "LASTPRINTEDDATE_FLG":          "s.LASTPRINTEDDATE_FLG",
    "ACCESSVALIDITYMODIFIEDDATE":   "s.ACCESSVALIDITYMODIFIEDDATE",
    "CREATEDDATE":                  "s.CREATEDDATE",
    "COMPANYPRODUCTCODE":           "s.COMPANYPRODUCTCODE",
    "PAYMENTSTATUS":                "s.PAYMENTSTATUS",
    "PHOTOKEY":                     "s.PHOTOKEY",
    "PHOTOSOURCE":                  "s.PHOTOSOURCE",
    "PHOTOSOURCETYPE":              "s.PHOTOSOURCETYPE",
    "PACKAGE_NAME":                 "s.PACKAGE_NAME",
    "W_INSERT_DT":                  F.current_timestamp(),
    "W_UPDATE_DT":                  F.current_timestamp(),
    "ETL_PROC_WID":                 F.lit(etl_proc_wid) # Add ETL_PROC_WID from parameter
}).execute()

## Optimize Target Table

In [ ]:
# Optimize target table after merge
spark.sql("SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false")
spark.sql("OPTIMIZE workspace.prxbi_dw.wc_badge_details_d ZORDER BY (integration_id, datasource_num_id)")

## Cleanup

In [ ]:
# SCEN_TASK_NO {180}, {210}
# Drop temporary tables
spark.sql("DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow")
spark.sql("DROP TABLE IF EXISTS workspace.prxbi_dw.c_mercury_badge_stg PURGE")

## Validation

In [ ]:
final_target_count = spark.table("workspace.prxbi_dw.wc_badge_details_d").count()
print(f"Final target table count: {final_target_count}")

print("Sample data from target table:")
display(spark.table("workspace.prxbi_dw.wc_badge_details_d").limit(10))

In [ ]:
spark.stop()

## Conversion Notes

- The ODI `MAX` self-join deduplication pattern for `WC_MERCURY_BADGE_TS` and `WC_BADGE_PRODUCT_D` has been converted to PySpark `Window.partitionBy().orderBy().row_number()` / `rank()` operations to select the latest/most relevant record.
- `NVL2` functions are replaced with `F.when(col.isNotNull(), ...).otherwise(...)`.
- `DBMS_STATS.GATHER_TABLE_STATS` calls are replaced by `OPTIMIZE` statements on the respective Delta tables.
- `CREATE INDEX` for flow tables is effectively handled by `OPTIMIZE ZORDER BY`.
- `ROW_WID` in the target table is assumed to be an `IDENTITY` column (based on `WC_BADGE_DETAILS_D_SEQ.NEXTVAL` in the original insert statement) and is therefore excluded from explicit inserts/updates in the `DeltaTable.merge()` statement. It is only included in the `whenNotMatchedInsert` values for `ETL_PROC_WID` to track the process that inserted the record.
- The hardcoded `DATASOURCE_NUM_ID = 380` has been replaced by the `datasource_num_id` widget parameter, with `380` as the default value.
- The `ETL_PROC_WID` is populated from the `WC_ETL_PARAMETERS` table if not provided via widget, and then used in the `whenNotMatchedInsert` clause.
- All table names and schema names are lowercased and prefixed with `workspace.` as per instructions.
- `NOLOGGING` and `/*+ append */` hints are removed as they are Oracle-specific.